# CELESTE Walkthrough

This tutorial assumes have installed CELESTE and its requirement packages and illustrates how to use the CELESTE library to develop your own unit and system models.
It aims to help you decode the library and get you up and running with the models quickly.

# Part 1 - Overview of the CELESTE Framework

The Framework is based on 3 core modules:
- base
- thermo
- systems

Part 2 will focus on making components based on Base and Thermo, while Part 3 will focus on Systems. 
The tutorial will be extended to include different related topics.

These core modules are then used to build unit and system models to form the CELESTE Standard Library (CSL) consisting of:
- components - Process components i.e. compressors and GPU
- electrochem - electrochemical reactor models i.e. SOC and AEL
- flow - flow control objects i.e. sources, sinks and mixers
- heat - Heat transfer objects i.e. HEX and Heaters
- interfaces - information transfer between objects i.e. Flange
- media - Fluid properties for specific mixtures using thermo
- reactors - Non-electrochemical reactor models i.e. methanation, HB etc.

# Part 2 - Components

These models in the CSL contain a specific structure given by ``ModelBase``

All unit models follow the structure:

```python
class UnitModel(ModelBase):
    def __init__(self, name, Medium, [other parameters]):
        super().__init__(locals())
        [initialise interfaces]

    def Eqs(self):
        self.Eq = [system of equations]
        return self.Eq

    def SomeOtherMethod(self): 
        return SomeOtherResult

    @property
    def someProperty
        [possible execution code here]
        return someValue

    def f_sol(self, x): 
        [unit variables that are unknown] = x[]

    def run_sol(self):
        x0 = [initial guess for unknowns]
        sol = solver(fun=self.f_sol, guess=x0)
        return sol

```

Here, `__init__()` and `Eqs()` are also found in `ModelBase` but were mostly empty and we add `SomeOtherMethod()`, `f_sol()`, and `run_sol()` on top of this to give the behaviour we want to model.

Lets look closer at what should be done for each method with the example of a mixer. We can Illustrate how a mixer should work as follows.
![mixer figure](misc/Mixer.png)

We have three streams, 2 inlet and one outlet therefore we need three flanges. 

```python

class Mixer(ModelBase):
    def __init__(self, name, Medium, [params to be defined]):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, [params F0])
        self.F1 = Flange(name + '.F1', Medium=Medium, [params F1])
        self.F2 = Flange(name + '.F2', Medium=Medium, [params F2])
```





Here we are using ModelBase and Flange which come from the CELESTE library and must be imported before mentioned:

In [1]:
from celeste.CSL import ModelBase
from celeste.CSL.interfaces import Flange
    

## Flange

Lets take a detour and zoom into the flange.

The Flange object takes the params `Flange(name, Medium, n, Xi, T, P)` 
- For `name` we related this to the name of the unit to give us `"mixer.F0"` as the name of the F0 flange.
- For the `Medium`, this comes from the `media` module described below
- `n` is the molar flow rate
- `Xi` the molar composition of each species
- `T` the temperature at the flange
- `P` the pressure at the flange


This is used in almost every object so its good to get comfortable with this object. Lets make a Flange object using the Air Medium:

In [2]:
import celeste.CSL.media as media
import numpy as np
f = Flange('f', Medium=media.Air(), n=1, Xi=np.array([0.21, 0.79]), T=300, P=1) # Normal Air flowing 1 mol/s @300K, 1bar

We can view the main detials of this flange by printing it:

In [3]:
print(f)

n /mol/s          1.000000
m /g/s           28.850334
T /K            300.000000
P /bar            1.000000
h /kJ/mol         0.053981
cp /kJ/mol/K      0.029180
Xi_O2             0.210000
Xi_N2             0.790000


This returns a pandas DataFrame of the impotant flow details.
We can also look into some thermodynamic properties such as enthalpy:

In [4]:
f.h #kJ/mol

np.float64(0.05398080388566727)

Other properties include:
- P: Pressure /bar
- T: Temperature /K
- Xi: Mole Fraction /-
- Yi: Mass Fraction /-
- cp: Heat Capacity /kJ/mol/K
- g: Gibbs Free Energy /kJ/mol
- h: Enthalpy /kJ/mol
- mw: Molar Mass /g/mol
- s: Entropy /kJ/mol/K

We can also change the T, P, and/or Xi of the flange after initialising it and it will automatically recalculate the properties:

In [5]:
f.T = 900
f.h

np.float64(18.437032884433545)

In this way the flange not only becomes and interfaces between objects, but also one for conveniently calculating thermodynamic properties for specific mixtures based on the media object.

## Media

Let's continue the detour and quickly look into the media package.
The core CSL contains the following media:

- ``Medium_Base(name, species)`` - Contains the core infrastructure for all media models
- ``Medium_NASAbase(name, species, T, P, Xi)`` - Contains core infrastructure for media models using NASA polynomial thermo properties
    - ``Air['O2','N2']`` - Simple Air model
    - ``Fuel_CH4['H2', 'CH4', 'CO2', 'CO', 'H2O', 'N2']`` - For SOC and Downstream processes
    - ``Fuel_CH4L['H2', 'CH4', 'CO2', 'CO', 'H2O(L)', 'N2']`` - For considering condensed water phase
    - ``Fuel_MeOH['H2', 'CH4', 'CO2', 'CO', 'H2O', 'N2', 'CH3OH']`` - For Methanol process
    - ``HB['H2','N2','NH3']`` - For Haber-Bosch process
    - ``Media_AEL['H2O(L)', 'H2', 'O2', 'H2O']`` - For AEL electrolyser systems
    - ``Steam['H2O]`` - Pure steam flows
    - ``Steam_Water['H2O', 'H2O(L)']`` - Two Phase water flows
    - ``Water['H2O(L)']`` - Pure liquid water flows

This approach allows for smaller systems of equations compared with loading all species.
This also allows for easy development of new media models based on different thermodynamic libraries i.e. cantera, coolprop etc.
The NASA properties data can be found in celeste/data. You can use the `Thermo.df` to see the different species and data conveniently as a pandas DataFrame.

In [6]:
from celeste.CSL.thermo import Thermo
print(f'Number of Species = {Thermo.df.index.size}')

Number of Species = 1895


Let's make our own media model for the Mixer:

In [7]:
class MediaExample(media.Medium_NASAbase):
    def __init__(self, name = 'MediaExample'):
        self._reference_X = np.array([0.25, 0.5, 0.25])  # needed for when you don't care about setting initial compositions
        super().__init__(name, species=np.array(['CO2', 'CO', 'CH4']) , Xi=self._reference_X)

Using this media in a flange would look like:

In [8]:
ExampleFlange = Flange('ExampleFlange', Medium=MediaExample()) # use defaults for n, Xi, T, P

In [9]:
print(ExampleFlange.Xi) # taken from reference_X
print(ExampleFlange.T, ExampleFlange.P )
print(ExampleFlange.h ) # Calculated at default conditions

[0.25 0.5  0.25]
298.15 1.0
-172.2941160036823


## Finishing Model

Now that we know how to use the ``Flange`` and ``media`` models, we can complete the ``__init__`` method by passing the necessary parameters in:

```python

class Mixer(ModelBase):
    def __init__(self, name, Medium, n_F0, Xi_F0, T_F0, P_F0, 
                                    n_F1, Xi_F1, T_F1, P_F1, 
                                    n_F2, Xi_F2, T_F2, P_F2):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, n=n_F0, Xi=Xi_F0, T=T_F0, P=P_F0)
        self.F1 = Flange(name + '.F1', Medium=Medium, n=n_F1, Xi=Xi_F1, T=T_F1, P=P_F1)
        self.F2 = Flange(name + '.F2', Medium=Medium, n=n_F2, Xi=Xi_F2, T=T_F2, P=P_F2)
```

Depending on the model, we can also include other variables that we deem constant or pre initialise some variables to avoid creating them later on for performance reasons.

We can then move on to including the system of equations needed to solve the model. These are usually done through balance equations of the form
`[Input] - [Output] + [Generation] - [Consumption] = 0`
- Starting with the mass balance: `F0 + F1 - F2 = 0`
- There is also a balance for the species: `X0_i*F0 + X1_i*F1 - X2_i*F2 = 0` for given component `i`
- Then the Energy balance: `h0*F0 + h1*F1 - h2*F2 = 0`
- Finally the pressure. Here we will simplify this to assign a pressure to the outlet stream based on F0: `P2 = P0`

This looks like the following:
```python

    def Eqs(self):
        self.Eq = np.concatenate((
            [self.F0.n + self.F1.n - self.F2.n],
            self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n - self.F2.Xi*self.F2.n,
            [self.F0.h*self.F0.n + self.F1.h*self.F1.n - self.F2.h*self.F2.n]
        ))
        self.F2.P = self.F0.P
        return self.Eq
```

Then we have to decide on the unknowns, which are `F2.n`, `F2.Xi`, and `F2.T` and we use this to build `f_sol`

```python

    def f_sol(self, x):
        self.F2.n = x[0]
        self.F2.Xi = x[1:self.Medium.nSpecies]
        self.F2.T = x[self.Medium.nSpecies]
        return self.Eqs() # update self.Eq based on x
```
Now we have formed the system of equations in a form that can be used by numerical solvers, where we give in a parameter `x` into `f_sol` and we get the errors of the system of equations from `self.Eq`. The solver will iteratively change `x`, until `self.Eq` is a vector of zeros (or close enough). In this case we wrap this functionality into the unit itself since we don't want to deal with giving guesses to all the different units and systems. So in this case we relate the initial guess to the input Flanges.

```python
    def run_sol(self):
        x0 = np.concatenate((
            [self.F0.n + self.F1.n],
            (self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n)/self.F2.n,
            self.F0.T
        ))
        sol = spo.root(self.f_sol, x0)
        return sol
```

For most cases we use `scipy.optimize.root()` for solving the system of equations. This library is usually imported as `import scipy.optimize as spo`. Another solver that is used in `spo` is `least_squares()` which is used in a similar way.

bringing it all together we get the following:

In [10]:
import scipy.optimize as spo

class Mixer(ModelBase):
    def __init__(self, name, Medium, n_F0, Xi_F0, T_F0, P_F0, 
                                    n_F1, Xi_F1, T_F1, P_F1, 
                                    n_F2, Xi_F2, T_F2, P_F2):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, n=n_F0, Xi=Xi_F0, T=T_F0, P=P_F0)
        self.F1 = Flange(name + '.F1', Medium=Medium, n=n_F1, Xi=Xi_F1, T=T_F1, P=P_F1)
        self.F2 = Flange(name + '.F2', Medium=Medium, n=n_F2, Xi=Xi_F2, T=T_F2, P=P_F2)

    def Eqs(self):
            self.Eq = np.concatenate((
                [self.F0.n + self.F1.n - self.F2.n],
                self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n - self.F2.Xi*self.F2.n,
                [self.F0.h*self.F0.n + self.F1.h*self.F1.n - self.F2.h*self.F2.n]
            ))
            self.F2.P = self.F0.P
            return self.Eq

    def f_sol(self, x):
            self.F2.n = x[0]
            self.F2.Xi = x[1:self.Medium.Ns+1]
            self.F2.T = x[self.Medium.Ns+1]
            return self.Eqs() 
        
    def run_sol(self):
        x0 = np.concatenate((
            [self.F0.n + self.F1.n],
            (self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n)/self.F2.n,
            [self.F0.T]
        ))
        sol = spo.root(self.f_sol, x0)
        return sol

Lets test this model:

In [11]:
mixer = Mixer('mixer', MediaExample(), n_F0 = 1, Xi_F0 = np.array([0, 1, 0]), T_F0 = 300, P_F0 = 1, 
              n_F1 = 1, Xi_F1 = np.array([1, 0,0]), T_F1 = 400, P_F1 = 1, 
              n_F2 = 1, Xi_F2 = None, T_F2 = 400, P_F2 = 1)

We can see the residulas based on the initial values we set:

In [12]:
mixer.Eqs()

array([ 1.00000000e+00,  7.50000000e-01,  5.00000000e-01, -2.50000000e-01,
       -3.31146952e+02])

And then running the system of equations:

In [13]:
mixer.run_sol()

 message: The solution converged.
 success: True
  status: 1
     fun: [ 0.000e+00  0.000e+00  0.000e+00 -6.209e-29  5.684e-14]
       x: [ 2.000e+00  5.000e-01  5.000e-01  3.104e-29  3.581e+02]
  method: hybr
    nfev: 14
    fjac: [[-1.984e-03 -1.984e-03 ...  2.367e-30  1.000e+00]
           [ 6.930e-01 -1.987e-01 ...  1.093e-19  2.356e-03]
           ...
           [ 8.622e-04  1.734e-01 ... -9.836e-01  4.428e-04]
           [-4.707e-03 -9.468e-01 ... -1.802e-01 -2.417e-03]]
       r: [ 5.039e+02  7.833e+02 ... -3.053e-05  1.667e-04]
     qtf: [ 1.435e-09  3.381e-12  6.765e-13  6.355e-13 -3.469e-12]

We see that the system solved successfully!
Now let's confirm the results make sense by looking into the outlet Flange

In [14]:
mixer.F2

n /mol/s        2.000000e+00
m /g/s          7.201960e+01
T /K            3.581443e+02
P /bar          1.000000e+00
h /kJ/mol      -2.499924e+02
cp /kJ/mol/K    3.447237e-02
Xi_CO2          5.000000e-01
Xi_CO           5.000000e-01
Xi_CH4          3.104268e-29

We can store this model in our own library to reuse the component without copy pasting everytime

## Building Your Own Component Library

It is not recommended to alter the code in the CSL. Instead, if you want to develop your own models, this should be done locally.
If you then want these models to be shared through the CSL, you can do this through the git issues.
Here we will discuss how you can build your own library and use them in your code.

First off, you need to make the folders where these components will be stored. This can be anywhere in your PC, but keep note of it. Here we will refer to this folder as `mylibrary`

So your folder structure would look like:
```tree
├──[My-CELESTE-Projects]
|    ├──mylibrary
|    |   ├── __init__.py
|    |   ├── mycomponents.py
|    |
|    ├──[Project Folder 1] 
|    ├──[Project Folder 2] 
|    ├──[Project Folder ...]
|    ├──[Project Folder N]
|
└──[Other Files]
```

The `__init__.py` is just an empty text file and is needed to tell python that `mylibrary` is a python module and all files ending in `.py` are submodules that can be imported in other python programmes.

Once you have these folders, you can paste your CELESTE objects into `mycomponents.py` along with any imports needed. So `mycomponents.py` should look like the following:


```python

import scipy.optimize as spo
from celeste.CSL.interfaces import Flange
import celeste.CSL.media as media
import numpy as np

class MediaExample(media.Medium_NASAbase):
    def __init__(self, name = 'MediaExample'):
        self._reference_X = np.array([0.25, 0.5, 0.25])  # needed for when you don't care about setting initial compositions
        super().__init__(name, species=np.array(['CO2', 'CO', 'CH4']) , Xi=self._reference_X)

class Mixer(ModelBase):
    def __init__(self, name, Medium, n_F0, Xi_F0, T_F0, P_F0, 
                                    n_F1, Xi_F1, T_F1, P_F1, 
                                    n_F2, Xi_F2, T_F2, P_F2):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, n=n_F0, Xi=Xi_F0, T=T_F0, P=P_F0)
        self.F1 = Flange(name + '.F1', Medium=Medium, n=n_F1, Xi=Xi_F1, T=T_F1, P=P_F1)
        self.F2 = Flange(name + '.F2', Medium=Medium, n=n_F2, Xi=Xi_F2, T=T_F2, P=P_F2)

    def Eqs(self):
            self.Eq = np.concatenate((
                [self.F0.n + self.F1.n - self.F2.n],
                self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n - self.F2.Xi*self.F2.n,
                [self.F0.h*self.F0.n + self.F1.h*self.F1.n - self.F2.h*self.F2.n]
            ))
            self.F2.P = self.F0.P
            return self.Eq

    def f_sol(self, x):
            self.F2.n = x[0]
            self.F2.Xi = x[1:self.Medium.Ns+1]
            self.F2.T = x[self.Medium.Ns+1]
            return self.Eqs() 
        
    def run_sol(self):
        x0 = np.concatenate((
            [self.F0.n + self.F1.n],
            (self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n)/self.F2.n,
            [self.F0.T]
        ))
        sol = spo.root(self.f_sol, x0)
        return sol

```

To be able to import this code into your projects, Python needs to know where this folder is. There are two approaches we can take to achieve this:
1. Using a .pth file
2. Making a pip installable package of your component library

Both approaches allow you to continuously build your library of components.

In the following setion we will outline both ways.

### .pth File Approach

1. Create a text file and change the extension from `.txt` to ``.pth`` i.e. `mycelestelibrary.pth` (like Tutorial/Example.pth)
2. In this file paste in the complete path to `[My-CELESTE-Projects]`

[Python]
3. Now open a command prompt and write `where python` and open the folder this `python.exe` file exists in.

[conda]
3. Open conda prompt and write `conda env list`. Open the foler of the environment you want to work in

4. Go into the folders Lib/site-packages/
5. Paste the `mycelestelibrary.pth`
6. Test that this works by opening a python terminal (open cmd or conda prompt and type `python` and enter) then `import mylibrary` you should see no error.

### Pip Installable Package Approach

1. Ccopy the two text files, `pyproject.toml`(i.e. Tutorial\example_pyproject.toml) and `setup.cfg`(i.e. Tutorial\example_setup.cfg) and paste into the root of `[My-CELESTE-Projects]`
2. In `setup.cfg` fill in the necessary data
3. Open your python/conda terminal and `cd [path to MY-CELESTE-projects]`
4. then pip install -e . to perform an editable install of your library.
5. If everything went well you should see success.
6. To test, in the terminal open python then `import mylibrary`. You should see no errors

# Part 3 - Systems

For this section we will use the components in the CSL. But the procedure is the same with your own library.

We will use the folder tree structure already established and work in the folder `Project Folder 1`

Here we will make a file called `projectsystem.py` and a jupyter notebook called `runprojectsystem.ipynb`.

The `.py` file will contain our system which extends from `celeste.CSL.SysBase` and the `.ipynb` file will run and explore the results and make plots etc.

A system describes its flowsheet: the units are created in `__init__` and every stream between them is declared once with `self.link(outlet, inlet)`. From these links CELESTE works out the order in which the units are solved, finds any recycle loops and chooses the tear streams to converge them - `run_sol()` then solves the whole system. (A system can still write its own `seq_sol` that connects the units in order with `connect_seq`, for example for an unusual solve sequence.)

in the `.py` file we will make a system model that looks like the following

```python
from celeste.CSL import SysBase
class ProjSys(SysBase):
    def __init__(self, [parameters]):
        super().__init__(locals())
        # Initialise components with parameters
        self.source1 = Source(params)
        self.component1 = Component1(params)
        self.component2 = Component2(params)
        self.sink1 = Sink(params)
        [Other components]

        # The streams of the flowsheet, each declared once (the order does not matter)
        self.link(self.source1.F1, self.component1.F0)
        self.link(self.component1.F1, self.component2.F0)
        self.link(self.component2.F1, self.sink1.F0)
        [other streams]
```

and in the notebook

```python
sys = ProjSys([parameters])
sys.run_sol()                                         # solve
sys.add_spec('sink1.F0.n', 2.0, vary='source1.F1.n')  # design specification: adjust an input until a result has a value
sys.solve_specs()
```

We will use this template to build our own simple system model for the following PFD

In [15]:
import numpy as np

from celeste.CSL import flow, components, reactors, interfaces, media, SysBase
class ExampleReformerSys(SysBase):
    def __init__(self, name, n0, X0, T, p):
        super().__init__(locals())
        # Initialise components with parameters
        self.source1 = flow.SourceM('source1', media.Fuel_CH4(), n0, X0, T, p)
        self.reformer = reactors.MR('reformer', media.Fuel_CH4(), T=T)
        self.sink1 = flow.Sink('sink1', media.Fuel_CH4())

        # The streams
        self.link(self.source1.F1, self.reformer.F0)
        self.link(self.reformer.F1, self.sink1.F0)

Now to use this model we create an instance

In [16]:
reformerSys = ExampleReformerSys('ReformerSys', 1, np.array([0.01, 0.3, 0.2, 1e-5, 0.5, 1e-5]), 973.15, 5.)

At this point the system is unsolved so if we were to see the flanges, the flow sheet would not make much sense:

In [17]:
reformerSys.ViewFlange([reformerSys.source1.F1, reformerSys.reformer.F0, reformerSys.reformer.F1, reformerSys.sink1.F0])

,source1((source1)).F1,reformer.F0,reformer.F1,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000e+00,1.000000e+00,1.000000e+00
m /g/s,22.642997,1.641534e+01,1.641534e+01,1.641534e+01
T /K,973.150000,9.731500e+02,9.731500e+02,2.981500e+02
P /bar,5.000000,2.300000e+01,2.300000e+01,1.000000e+00
h /kJ/mol,-191.940338,-1.932460e+02,-1.932460e+02,-2.176422e+02
cp /kJ/mol/K,0.053314,3.985894e-02,3.985894e-02,3.311239e-02
Xi_H2,0.010000,1.000000e-01,1.000000e-01,1.000000e-01
Xi_CH4,0.300000,1.000000e-08,1.000000e-08,1.000000e-08
Xi_CO2,0.200000,1.000000e-08,1.000000e-08,1.000000e-08
Xi_CO,0.000010,1.000000e-08,1.000000e-08,1.000000e-08


Before solving, we can ask CELESTE in which order it will solve the units (generated from the links):

In [18]:
reformerSys.print_solve_order();

1. source1((source1))
2. reformer
3. sink1[(sink1)]


So let's run this system

In [19]:
reformerSys.run_sol()

In [20]:
reformerSys.ViewFlange([reformerSys.source1.F1, reformerSys.reformer.F0, reformerSys.reformer.F1, reformerSys.sink1.F0])

,source1((source1)).F1,reformer.F0,reformer.F1,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000,1.384907,1.384907
m /g/s,22.642997,22.642997,22.642997,22.642997
T /K,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.863966,-107.863966
cp /kJ/mol/K,0.053314,0.053314,0.039745,0.039745
Xi_H2,0.010000,0.010000,0.404089,0.404089
Xi_CH4,0.300000,0.300000,0.081274,0.081274
Xi_CO2,0.200000,0.200000,0.135241,0.135241
Xi_CO,0.000010,0.000010,0.144528,0.144528


we can verify that the system solved properly in two ways:
1) check the mass balance of a process unit if the `MBcheck()` method exists, or 
2) Directly look at the solver results if they exist for the unit with `UNIT._sol`

In [21]:
reformerSys.reformer.MBcheck()

np.float64(0.0)

In [22]:
reformerSys.reformer._sol

       message: The solution converged.
       success: True
        status: 1
           fun: [ 0.000e+00  0.000e+00  0.000e+00 -3.388e-16  1.665e-16
                  4.441e-16]
             x: [-5.805e-01 -2.184e+00 -1.675e+00 -1.609e+00 -1.123e+00
                 -1.151e+01]
        method: hybr
          nfev: 25
          fjac: [[ 9.944e-02 -2.578e-02 ... -3.249e-01 -9.391e-01]
                 [-3.995e-01 -5.667e-01 ... -6.889e-01  2.110e-01]
                 ...
                 [-5.851e-01  4.195e-01 ... -2.943e-02 -3.111e-02]
                 [ 1.019e-03 -1.308e-03 ... -1.578e-04 -3.557e-05]]
             r: [-3.080e+00  1.359e+00 ...  1.711e-03  1.000e+00]
           qtf: [-7.541e-15  3.965e-15 -1.207e-15  1.905e-15 -4.886e-15
                 -3.291e-16]
 residual_norm: 4.440892098500626e-16

In both cases the results show the system solved successfully.

## Looking at the flowsheet

The links are enough to draw the flowsheet. `.create_graph()` returns it as a Mermaid graph (`save=False` returns the text without writing a `.mmd` file). Let's give it a try

In [23]:
print(reformerSys.create_graph(save=False))

flowchart LR
source1((source1))
reformer
sink1[(sink1)]
	source1((source1)) --F1_F0--> reformer
	reformer --F1_F0--> sink1[(sink1)]



which when executed in a markdown executor that can evaluate mermaidJS (such as jupyter notebooks ;) ) you get the following

```mermaid
flowchart LR
source1((source1)) --F1_F0--> reformer
reformer --F1_F0--> sink1[(sink1)]
```

Notice that the name of the units is the same as that of the system attributes - this is because the names of the units in the strings were purposely made the same as the object in python itself - do this! it will help with any future debugging
- TL;DR - give your process units the same python name as the string i.e. `self.source1 = flow.SourceM('source1', ...)` and NOT `self.source1 = flow.SourceM('Source', ...)`

`.pfd()` draws an interactive process flow diagram instead: hover over a stream to see its state and composition, or over a unit to see its duties (in a notebook it displays itself, `.pfd('file.html')` also saves it).

In [24]:
reformerSys.pfd()

And `.print_equations()` shows what each unit solves: the reformer solves a system of equations (element balances and the shift and reforming equilibria) while the source and sink are algorithmic - they assign their outlet from their inlet.

In [25]:
reformerSys.print_equations();

ReformerSys: 3 units, 1 solve a system of equations (6 equations), the rest are algorithmic
  reformer (MR): 6 equations in 6 unknowns, max |residual| 4.44e-16
      Isothermal: element balances, WGS and MSR equilibrium.
  sink1 (Sink): outlet assigned from the inlet
  source1 (SourceM): outlet assigned from the inlet
streams declared with link(): 2 (copied between the units in the solve order)


## Design specifications
Now we have a small system, lets use it to answer an inverse question. We don't want to solve the system for a given inlet flow, we want the model to tell us what inlet flowrate is needed to get a desired outlet flow of 2 mol/s (pretty simple example, but other inverse problems follow the same principle).

A design specification says "adjust this input until that result has this value". Targets and inputs are written as paths relative to the system (a target can also be a function of the system).

In [26]:
reformerSys.add_spec('sink1.F0.n', 2.0, vary='source1.F1.n', bounds=(0.1, 10.0))
sol = reformerSys.solve_specs()
sol.success

True

In [27]:
reformerSys.specs

,spec,value,current,vary,input,lower,upper
0,sink1.F0.n,2.0,2.0,source1.F1.n,1.444141,0.1,10.0


### Okay... so what did we do here?

pretty simple:
1. `add_spec` declared the target (`sink1.F0.n` = 2 mol/s) and the input that is adjusted (`source1.F1.n`), with bounds.
2. `solve_specs` runs a root finder (`celeste.solvers.solve`, built on scipy.optimize `root` and `least_squares`) on the residual target - value. Every iteration is a full `run_sol` of the system, so recycles inside the system are converged each time.
3. `specs` shows the target, its current value and the adjusted input - the solution converged to high accuracy and quickly!

Several specifications are solved together (one input per specification). You can still write your own residual function and give it to `scipy.optimize` directly if you need something special.

Let's see what this looks like in our flanges

In [28]:
reformerSys.ViewFlange([reformerSys.source1.F1, reformerSys.reformer.F0, reformerSys.reformer.F1, reformerSys.sink1.F0])

,source1((source1)).F1,reformer.F0,reformer.F1,sink1[(sink1)].F0
n /mol/s,1.444141,1.444141,2.000000,2.000000
m /g/s,32.699671,32.699671,32.699671,32.699671
T /K,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.863966,-107.863966
cp /kJ/mol/K,0.053314,0.053314,0.039745,0.039745
Xi_H2,0.010000,0.010000,0.404089,0.404089
Xi_CH4,0.300000,0.300000,0.081274,0.081274
Xi_CO2,0.200000,0.200000,0.135241,0.135241
Xi_CO,0.000010,0.000010,0.144528,0.144528


So clearly we got what we wanted!

## What about recycles?

We can modify our system to consider recycles very easily. From an engineering point of view, this of course means that we need to add splitters and mixers to realise the recycle. In the model we only declare the recycle stream like any other stream - CELESTE finds the loop and converges it.

In [29]:
import numpy as np

from celeste.CSL import flow, components, reactors, interfaces, media, SysBase
class ExampleReformerRecycleSys(SysBase):
    def __init__(self, name, n0, X0, T, p):
        super().__init__(locals())
        # Initialise components with parameters
        self.source1 = flow.SourceM('source1', media.Fuel_CH4(), n0, X0, T, p)
        self.reformer = reactors.MR('reformer', media.Fuel_CH4(), T=T)
        self.sink1 = flow.Sink('sink1', media.Fuel_CH4())
        self.split = flow.Splitter('split', media.Fuel_CH4(), Split=0.5)
        self.mix = flow.Mixer('mix', media.Fuel_CH4())
        self.mix.F1.n = 0                                       # first guess of the recycle: no flow

        # The streams
        self.link(self.source1.F1, self.mix.F0)                 # feed to the mixer
        self.link(self.mix.F2, self.reformer.F0)                # mixer to reformer
        self.link(self.reformer.F1, self.split.F0)              # reformer to splitter
        self.link(self.split.F1, self.sink1.F0)                 # product
        self.link(self.split.F2, self.mix.F1)                   # recycle

In [30]:
reformerSysRecycle = ExampleReformerRecycleSys('ReformerSys', 1, np.array([0.01, 0.3, 0.2, 1e-5, 0.5, 1e-5]), 973.15, 5.)

Let's have a look at the solve order CELESTE generated

In [31]:
reformerSysRecycle.print_solve_order();

1. source1((source1))
2. loop [mix, reformer, split]  tear: split.F2 -> mix.F1
3. sink1[(sink1)]


The units form one loop, and the recycle stream `split.F2 -> mix.F1` is the tear stream: its value is guessed (here: no flow), the loop is run, and the guess is updated until the computed recycle equals the guess (`Recycle_Tear`). Before solving, the flanges show only the first guess:

In [32]:
reformerSysRecycle.ViewFlange([reformerSysRecycle.source1.F1, 
                                reformerSysRecycle.mix.F0,
                                reformerSysRecycle.mix.F1,
                                reformerSysRecycle.mix.F2,
                                reformerSysRecycle.reformer.F0, 
                                reformerSysRecycle.reformer.F1, 
                                reformerSysRecycle.split.F0,
                                reformerSysRecycle.split.F1,
                                reformerSysRecycle.split.F2,
                                reformerSysRecycle.sink1.F0])

,source1((source1)).F1,mix.F0,mix.F1,mix.F2,reformer.F0,reformer.F1,split.F0,split.F1,split.F2,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
m /g/s,22.642997,1.641534e+01,0.000000e+00,1.641534e+01,1.641534e+01,1.641534e+01,1.641534e+01,1.641534e+01,1.641534e+01,1.641534e+01
T /K,973.150000,2.981500e+02,2.981500e+02,2.981500e+02,9.731500e+02,9.731500e+02,2.981500e+02,2.981500e+02,2.981500e+02,2.981500e+02
P /bar,5.000000,1.000000e+00,1.000000e+00,1.000000e+00,2.300000e+01,2.300000e+01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
h /kJ/mol,-191.940338,-2.176422e+02,-2.176422e+02,-2.176422e+02,-1.932460e+02,-1.932460e+02,-2.176422e+02,-2.176422e+02,-2.176422e+02,-2.176422e+02
cp /kJ/mol/K,0.053314,3.311239e-02,3.311239e-02,3.311239e-02,3.985894e-02,3.985894e-02,3.311239e-02,3.311239e-02,3.311239e-02,3.311239e-02
Xi_H2,0.010000,1.000000e-01,1.000000e-01,1.000000e-01,1.000000e-01,1.000000e-01,1.000000e-01,1.000000e-01,1.000000e-01,1.000000e-01
Xi_CH4,0.300000,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08
Xi_CO2,0.200000,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08
Xi_CO,0.000010,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08,1.000000e-08


`run_sol()` solves the system with its recycle:

In [33]:
reformerSysRecycle.run_sol()

In [34]:
reformerSysRecycle.ViewFlange([reformerSysRecycle.source1.F1, 
                                reformerSysRecycle.mix.F0,
                                reformerSysRecycle.mix.F1,
                                reformerSysRecycle.mix.F2,
                                reformerSysRecycle.reformer.F0, 
                                reformerSysRecycle.reformer.F1, 
                                reformerSysRecycle.split.F0,
                                reformerSysRecycle.split.F1,
                                reformerSysRecycle.split.F2,
                                reformerSysRecycle.sink1.F0])

,source1((source1)).F1,mix.F0,mix.F1,mix.F2,reformer.F0,reformer.F1,split.F0,split.F1,split.F2,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000,1.384907,2.384907,2.384907,2.769813,2.769813,1.384907,1.384907,1.384907
m /g/s,22.642997,22.642997,22.642997,45.285994,45.285994,45.285994,45.285994,22.642997,22.642997,22.642997
T /K,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.863966,-143.117492,-143.117492,-107.863966,-107.863966,-107.863966,-107.863966,-107.863966
cp /kJ/mol/K,0.053314,0.053314,0.039745,0.045434,0.045434,0.039745,0.039745,0.039745,0.039745,0.039745
Xi_H2,0.010000,0.010000,0.404089,0.238846,0.238846,0.404089,0.404089,0.404089,0.404089,0.404089
Xi_CH4,0.300000,0.300000,0.081274,0.172986,0.172986,0.081274,0.081274,0.081274,0.081274,0.081274
Xi_CO2,0.200000,0.200000,0.135241,0.162395,0.162395,0.135241,0.135241,0.135241,0.135241,0.135241
Xi_CO,0.000010,0.000010,0.144528,0.083931,0.083931,0.144528,0.144528,0.144528,0.144528,0.144528


It was able to solve the recycle very quickly, with very few iterations. In the diagram the recycle stream is dashed:

In [35]:
reformerSysRecycle.pfd()

Now let's see if it can handle the design specification with the recycle - every solver iteration converges the recycle again

In [36]:
reformerSysRecycle.add_spec('sink1.F0.n', 2.0, vary='source1.F1.n', bounds=(0.1, 10.0))
sol_rec = reformerSysRecycle.solve_specs()
reformerSysRecycle.specs

,spec,value,current,vary,input,lower,upper
0,sink1.F0.n,2.0,2.0,source1.F1.n,1.444141,0.1,10.0


In [37]:
reformerSysRecycle.ViewFlange([reformerSysRecycle.source1.F1, 
                                reformerSysRecycle.mix.F0,
                                reformerSysRecycle.mix.F1,
                                reformerSysRecycle.mix.F2,
                                reformerSysRecycle.reformer.F0, 
                                reformerSysRecycle.reformer.F1, 
                                reformerSysRecycle.split.F0,
                                reformerSysRecycle.split.F1,
                                reformerSysRecycle.split.F2,
                                reformerSysRecycle.sink1.F0])

,source1((source1)).F1,mix.F0,mix.F1,mix.F2,reformer.F0,reformer.F1,split.F0,split.F1,split.F2,sink1[(sink1)].F0
n /mol/s,1.444141,1.444141,2.000000,3.444141,3.444141,4.000000,4.000000,2.000000,2.000000,2.000000
m /g/s,32.699671,32.699671,32.699671,65.399342,65.399342,65.399342,65.399342,32.699671,32.699671,32.699671
T /K,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.863966,-143.117492,-143.117492,-107.863966,-107.863966,-107.863966,-107.863966,-107.863966
cp /kJ/mol/K,0.053314,0.053314,0.039745,0.045434,0.045434,0.039745,0.039745,0.039745,0.039745,0.039745
Xi_H2,0.010000,0.010000,0.404089,0.238846,0.238846,0.404089,0.404089,0.404089,0.404089,0.404089
Xi_CH4,0.300000,0.300000,0.081274,0.172986,0.172986,0.081274,0.081274,0.081274,0.081274,0.081274
Xi_CO2,0.200000,0.200000,0.135241,0.162395,0.162395,0.135241,0.135241,0.135241,0.135241,0.135241
Xi_CO,0.000010,0.000010,0.144528,0.083931,0.083931,0.144528,0.144528,0.144528,0.144528,0.144528


The specification is met: the product flow is 2 mol/s with the recycle converged.

# Fin.

With this you should be equipped enough to start playing around with your own models and understand the models we have already made for the units and systems (or to ask us for help with your models)